In [ ]:
# train_models.py
import os
import shutil
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# ========= CONFIG =========
BASE_DIR = Path('/content/drive/MyDrive/Driver_Drowsiness_ and_Distraction_Detection_Dataset/train') # Corrected path
print(f"Updated BASE_DIR: {BASE_DIR}") # Added print statement
TEMP_DATA_DIR = Path('dataset')
MODELS_DIR = Path('models')
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 8
LEARNING_RATE = 1e-4
# ==========================

def check_folders():
    print("\n📂 Checking dataset folders...")
    for folder in ['Open', 'Closed', 'yawn', 'no_yawn']:
        p = BASE_DIR / folder
        if p.exists():
            count = len(list(p.glob('*.*')))
            print(f"✅ {folder}: {count} images")
        else:
            print(f"❌ Missing folder: {folder}")

def prepare_structure():
    eyes_dir = TEMP_DATA_DIR / 'eyes'
    mouth_dir = TEMP_DATA_DIR / 'mouth'
    for d in [eyes_dir / 'Open', eyes_dir / 'Closed', mouth_dir / 'yawn', mouth_dir / 'no_yawn']:
        d.mkdir(parents=True, exist_ok=True)

    def copy_files(src_folder, dest_folder):
        src = BASE_DIR / src_folder
        dest = TEMP_DATA_DIR / dest_folder
        for f in src.glob('*.*'):
            shutil.copy(f, dest / f.name)

    copy_files('Open', 'eyes/Open')
    copy_files('Closed', 'eyes/Closed')
    copy_files('yawn', 'mouth/yawn')
    copy_files('no_yawn', 'mouth/no_yawn')
    print("✅ Dataset structured successfully.\n")

def build_model():
    base = MobileNetV2(weights='imagenet', include_top=False,
                       input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.25)(x)
    output = Dense(1, activation='sigmoid')(x)
    model = Model(inputs=base.input, outputs=output)
    return model

def train_binary_model(data_root):
    datagen = ImageDataGenerator(
        rescale=1./255,
        validation_split=0.15,
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.1,
        zoom_range=0.15,
        horizontal_flip=True,
        brightness_range=(0.7, 1.2)
    )

    train_gen = datagen.flow_from_directory(
        str(data_root),
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='binary',
        subset='training'
    )
    val_gen = datagen.flow_from_directory(
        str(data_root),
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='binary',
        subset='validation'
    )

    print("🔠 Class mapping:", train_gen.class_indices)
    model = build_model()
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                  loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS)
    return model

def main():
    print("✅ Starting training script...")
    check_folders()
    prepare_structure()

    MODELS_DIR.mkdir(exist_ok=True)

    print("🚀 Training Eye Model (Open vs Closed)...")
    eye_model = train_binary_model(TEMP_DATA_DIR / 'eyes')
    eye_model.save(MODELS_DIR / 'eye_model.h5')
    print("✅ Eye model saved!\n")

    print("🚀 Training Mouth Model (yawn vs no_yawn)...")
    mouth_model = train_binary_model(TEMP_DATA_DIR / 'mouth')
    mouth_model.save(MODELS_DIR / 'mouth_model.h5')
    print("✅ Mouth model saved!\n")

    print("🎉 Training complete! Models stored in ./models/")

if __name__ == "__main__":
    main()

Updated BASE_DIR: /content/drive/MyDrive/Driver_Drowsiness_ and_Distraction_Detection_Dataset/train
✅ Starting training script...

📂 Checking dataset folders...
✅ Open: 632 images
✅ Closed: 617 images
✅ yawn: 617 images
✅ no_yawn: 367 images
✅ Dataset structured successfully.

🚀 Training Eye Model (Open vs Closed)...
Found 1063 images belonging to 2 classes.
Found 186 images belonging to 2 classes.
🔠 Class mapping: {'Closed': 0, 'Open': 1}
Epoch 1/8
67/67 ━━━━━━━━━━━━━━━━━━━━ 88s 1s/step - accuracy: 0.6940 - loss: 0.5742 - val_accuracy: 0.9409 - val_loss: 0.2017
Epoch 2/8
67/67 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.9532 - loss: 0.1748 - val_accuracy: 0.9409 - val_loss: 0.1449
Epoch 3/8
67/67 ━━━━━━━━━━━━━━━━━━━━ 85s 1s/step - accuracy: 0.9565 - loss: 0.1291 - val_accuracy: 0.9624 - val_loss: 0.1048
Epoch 4/8
67/67 ━━━━━━━━━━━━━━━━━━━━ 76s 1s/step - accuracy: 0.9665 - loss: 0.0956 - val_accuracy: 0.9570 - val_loss: 0.1171
Epoch 5/8
67/67 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accur

✅ Eye model saved!

🚀 Training Mouth Model (yawn vs no_yawn)...
Found 837 images belonging to 2 classes.
Found 147 images belonging to 2 classes.
🔠 Class mapping: {'no_yawn': 0, 'yawn': 1}
Epoch 1/8
53/53 ━━━━━━━━━━━━━━━━━━━━ 72s 1s/step - accuracy: 0.5849 - loss: 0.7551 - val_accuracy: 0.6803 - val_loss: 0.5487
Epoch 2/8
53/53 ━━━━━━━━━━━━━━━━━━━━ 78s 1s/step - accuracy: 0.6564 - loss: 0.6010 - val_accuracy: 0.6327 - val_loss: 0.5447
Epoch 3/8
53/53 ━━━━━━━━━━━━━━━━━━━━ 71s 1s/step - accuracy: 0.6567 - loss: 0.6081 - val_accuracy: 0.7347 - val_loss: 0.5099
Epoch 4/8
53/53 ━━━━━━━━━━━━━━━━━━━━ 63s 1s/step - accuracy: 0.7370 - loss: 0.5380 - val_accuracy: 0.6395 - val_loss: 0.5607
Epoch 5/8
53/53 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.7129 - loss: 0.5243 - val_accuracy: 0.6531 - val_loss: 0.5510
Epoch 6/8
53/53 ━━━━━━━━━━━━━━━━━━━━ 63s 1s/step - accuracy: 0.7041 - loss: 0.5154 - val_accuracy: 0.6667 - val_loss: 0.5713
Epoch 7/8
53/53 ━━━━━━━━━━━━━━━━━━━━ 63s 1s/step - accuracy: 

✅ Mouth model saved!

🎉 Training complete! Models stored in ./models/


In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 95.9 MB/s eta 0:00:00


In [ ]:
# app.py
import streamlit as st
from PIL import Image
import numpy as np
import tensorflow as tf
import os

EYE_MODEL_PATH = 'models/eye_model.h5'
MOUTH_MODEL_PATH = 'models/mouth_model.h5'
IMG_SIZE = (224, 224)

@st.cache_resource
def load_models():
    eye_model = tf.keras.models.load_model(EYE_MODEL_PATH)
    mouth_model = tf.keras.models.load_model(MOUTH_MODEL_PATH)
    return eye_model, mouth_model

def preprocess(img):
    img = img.convert('RGB').resize(IMG_SIZE)
    arr = np.array(img) / 255.0
    return np.expand_dims(arr, axis=0)

def predict(img, eye_model, mouth_model):
    x = preprocess(img)
    eye_pred = float(eye_model.predict(x, verbose=0)[0][0])
    mouth_pred = float(mouth_model.predict(x, verbose=0)[0][0])
    return eye_pred, mouth_pred

def classify(eye_pred, mouth_pred, threshold=0.5):
    eye_closed = eye_pred > threshold
    yawning = mouth_pred > threshold

    if eye_closed and yawning:
        return "😴 Drowsy (Eyes closed + Yawning)", "danger"
    elif eye_closed:
        return "😵 Drowsy (Eyes closed)", "danger"
    elif yawning:
        return "🥱 Yawning / Tired", "warning"
    else:
        return "✅ Alert and Attentive", "safe"

def display(status, level):
    colors = {"danger": "#ffcccc", "warning": "#fff2cc", "safe": "#ccffcc"}
    st.markdown(
        f"<div style='background-color:{colors[level]};padding:15px;border-radius:10px;text-align:center;font-size:18px;'>"
        f"<b>{status}</b></div>",
        unsafe_allow_html=True
    )

def main():
    st.set_page_config(page_title="Driver Drowsiness Detection", layout="centered")
    st.title("🚗 Driver Drowsiness & Distraction Detection")
    st.write("Upload an image or use your webcam to detect if the driver is alert, yawning, or drowsy.")

    if not (os.path.exists(EYE_MODEL_PATH) and os.path.exists(MOUTH_MODEL_PATH)):
        st.error("❌ Models not found. Please run train_models.py first.")
        st.stop()

    eye_model, mouth_model = load_models()
    choice = st.radio("Select Input Type", ["Upload Image", "Use Webcam"])

    img = None
    if choice == "Upload Image":
        file = st.file_uploader("Upload an image", type=["jpg", "jpeg", "png"])
        if file:
            img = Image.open(file)
            st.image(img, caption="Uploaded Image", use_column_width=True)
    else:
        cam = st.camera_input("Capture Image")
        if cam:
            img = Image.open(cam)
            st.image(img, caption="Captured Image", use_column_width=True)

    if img:
        with st.spinner("Analyzing..."):
            eye_pred, mouth_pred = predict(img, eye_model, mouth_model)
            status, level = classify(eye_pred, mouth_pred)
            display(status, level)
            st.write(f"**Eye Model Output:** {eye_pred:.3f}")
            st.write(f"**Mouth Model Output:** {mouth_pred:.3f}")

if __name__ == "__main__":
    # This is a helper to write the app.py file when running in Colab
    if 'google.colab' in str(get_ipython()):
        with open('app.py', 'w') as f:
            f.write(
"""
import streamlit as st
from PIL import Image
import numpy as np
import tensorflow as tf
import os

EYE_MODEL_PATH = 'models/eye_model.h5'
MOUTH_MODEL_PATH = 'models/mouth_model.h5'
IMG_SIZE = (224, 224)

@st.cache_resource
def load_models():
    eye_model = tf.keras.models.load_model(EYE_MODEL_PATH)
    mouth_model = tf.keras.models.load_model(MOUTH_MODEL_PATH)
    return eye_model, mouth_model

def preprocess(img):
    img = img.convert('RGB').resize(IMG_SIZE)
    arr = np.array(img) / 255.0
    return np.expand_dims(arr, axis=0)

def predict(img, eye_model, mouth_model):
    x = preprocess(img)
    eye_pred = float(eye_model.predict(x, verbose=0)[0][0])
    mouth_pred = float(mouth_model.predict(x, verbose=0)[0][0])
    return eye_pred, mouth_pred

def classify(eye_pred, mouth_pred, threshold=0.5):
    eye_closed = eye_pred > threshold
    yawning = mouth_pred > threshold

    if eye_closed and yawning:
        return "😴 Drowsy (Eyes closed + Yawning)", "danger"
    elif eye_closed:
        return "😵 Drowsy (Eyes closed)", "danger"
    elif yawning:
        return "🥱 Yawning / Tired", "warning"
    else:
        return "✅ Alert and Attentive", "safe"

def display(status, level):
    colors = {"danger": "#ffcccc", "warning": "#fff2cc", "safe": "#ccffcc"}
    st.markdown(
        f"<div style='background-color:{colors[level]};padding:15px;border-radius:10px;text-align:center;font-size:18px;'>"
        f"<b>{status}</b></div>",
        unsafe_allow_html=True
    )

def main():
    st.set_page_config(page_title="Driver Drowsiness Detection", layout="centered")
    st.title("🚗 Driver Drowsiness & Distraction Detection")
    st.write("Upload an image or use your webcam to detect if the driver is alert, yawning, or drowsy.")

    if not (os.path.exists(EYE_MODEL_PATH) and os.path.exists(MOUTH_MODEL_PATH)):
        st.error("❌ Models not found. Please run train_models.py first.")
        st.stop()

    eye_model, mouth_model = load_models()
    choice = st.radio("Select Input Type", ["Upload Image", "Use Webcam"])

    img = None
    if choice == "Upload Image":
        file = st.file_uploader("Upload an image", type=["jpg", "jpeg", "png"])
        if file:
            img = Image.open(file)
            st.image(img, caption="Uploaded Image", use_column_width=True)
    else:
        cam = st.camera_input("Capture Image")
        if cam:
            img = Image.open(cam)
            st.image(img, caption="Captured Image", use_column_width=True)

    if img:
        with st.spinner("Analyzing..."):
            eye_pred, mouth_pred = predict(img, eye_model, mouth_model)
            status, level = classify(eye_pred, mouth_pred)
            display(status, level)
            st.write(f"**Eye Model Output:** {eye_pred:.3f}")
            st.write(f"**Mouth Model Output:** {mouth_pred:.3f}")

if __name__ == "__main__":
    main()
"""
            )
    main()

2025-10-31 09:54:21.010 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-31 09:54:21.013 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-31 09:54:21.014 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-31 09:54:21.015 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-31 09:54:21.017 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-31 09:54:21.020 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-31 09:54:21.020 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-31 09:54:21.021 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
!pip install pyngrok


In [ ]:
from pyngrok import ngrok
import threading, time
from google.colab import userdata

# Get ngrok authtoken from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print("Please add your ngrok authtoken to Colab secrets as 'NGROK_AUTH_TOKEN'")

def run_app():
    !streamlit run app.py --server.port 8501

thread = threading.Thread(target=run_app)
thread.start()

time.sleep(5)
# Check if authtoken is set before attempting to connect
if NGROK_AUTH_TOKEN:
    public_url = ngrok.connect(8501)
    print("🌐 Your Streamlit app is live here:", public_url)
else:
    print("Cannot start ngrok tunnel without authtoken.")




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.123.151.6:8501

🌐 Your Streamlit app is live here: NgrokTunnel: "https://unconsignable-patrice-truthfully.ngrok-free.dev" -> "http://localhost:8501"
